# Sprint 5 – Notebook 11 – Handling Imbalanced Data



In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.utils import resample

X, y = make_classification(n_samples=240, n_features=5, n_informative=3,
                           n_redundant=0, weights=[0.9, 0.1], random_state=42)
X = pd.DataFrame(X, columns=[f"Feature_{i}" for i in range(1, 6)])
y = pd.Series(y, name="Fraud")
print('Class counts:', y.value_counts().to_dict())

Class counts: {0: 216, 1: 24}


## 1. What is Class Imbalance?

### Definition
Class imbalance means one target class appears much more often than another. The larger group is called the majority class. The smaller group is called the minority class. This is common in rare-event prediction.

### Business Example
A bank may have thousands of genuine transactions and only a few fraudulent ones. Fraud is important despite its low frequency. The raw dataset therefore has unequal class counts. This imbalance should be checked before modeling.

### AI/ML Use
A classifier can favor the majority class. Minority cases may be missed even when overall accuracy looks good. Class counts should be inspected before training. Recall and precision help reveal minority-class performance.

### Python Example

In [2]:
print(y.value_counts())
print((y.value_counts(normalize=True)*100).round(1))

Fraud
0    216
1     24
Name: count, dtype: int64
Fraud
0    90.0
1    10.0
Name: proportion, dtype: float64


### Explanation
The counts show how many examples belong to each class. The percentages make the imbalance easy to compare. Class 0 is the majority in this sample. Class 1 needs special attention during evaluation.

## 2. Balanced vs Imbalanced Dataset

### Definition
A balanced dataset has similar counts across classes. An imbalanced dataset has unequal class frequencies. Perfectly equal counts are not always necessary. The severity depends on the problem and the cost of errors.

### Business Example
A product survey with equal positive and negative responses is balanced. Fraud data with very few fraud cases is imbalanced. Both datasets can be useful. The appropriate evaluation method may differ.

### AI/ML Use
Balanced data often makes minority patterns easier to learn. Imbalanced data may require class weighting or resampling. Always keep a realistic test set. Compare class-wise metrics, not just total accuracy.

### Python Example

In [3]:
balanced = pd.Series([0]*10 + [1]*10)
imbalanced = pd.Series([0]*18 + [1]*2)
print("Balanced:", balanced.value_counts().to_dict())
print("Imbalanced:", imbalanced.value_counts().to_dict())

Balanced: {0: 10, 1: 10}
Imbalanced: {0: 18, 1: 2}


### Explanation
The first series contains ten examples per class. The second contains eighteen of class 0 and two of class 1. The second dataset is strongly imbalanced. Comparing counts is the simplest first check.

## 3. Why Class Imbalance is a Problem

### Definition
A model may learn to predict the frequent class most of the time. This can hide poor detection of rare events. Missing rare events may be expensive. Accuracy alone may not reflect this risk.

### Business Example
Suppose only 10 out of 1000 payments are fraudulent. A model that predicts every payment as genuine gets 99% accuracy. However, it catches no fraud. Such a model is not useful for fraud detection.

### AI/ML Use
Use precision, recall, F1, confusion matrices, and balanced accuracy. Consider the business cost of false negatives and false positives. Evaluate performance on untouched test data. Choose the threshold based on the real goal.

### Python Example

In [7]:
actual = np.array([0]*95 + [1]*5)
predicted = np.zeros(100, dtype=int)
print("Accuracy:", accuracy_score(actual, predicted))
print("Balanced accuracy:", balanced_accuracy_score(actual, predicted))

Accuracy: 0.95
Balanced accuracy: 0.5


### Explanation
Predicting only class 0 gives 95% overall accuracy. Yet the model finds none of the five minority examples. Balanced accuracy exposes this weakness. This is why accuracy should not be the only metric.

## 4. Undersampling

### Definition
Undersampling reduces the number of majority-class examples. It can create a more balanced training set. It is fast and requires no synthetic data. However, it may discard useful observations.

### Business Example
A retailer has many ordinary orders and few returned orders. It can keep a sample of ordinary orders for model training. This makes returns more visible during learning. Some ordinary-order patterns may be lost.

### AI/ML Use
Use undersampling when the majority class is very large. Apply it only to the training partition. Test on the original class distribution. Compare results against an unmodified baseline.

### Python Example

In [9]:
majority_ids = y[y == 0].index
minority_ids = y[y == 1].index
chosen = np.random.default_rng(42).choice(majority_ids, len(minority_ids), replace=False)
print(y.loc[list(chosen) + list(minority_ids)].value_counts())

Fraud
0    24
1    24
Name: count, dtype: int64


### Explanation
The code selects a smaller subset of majority cases. All minority examples are kept in this demonstration. The resulting counts are equal. In a real project, sample only after splitting the dataset.

## 5. Oversampling

### Definition
Oversampling increases minority-class representation. It may repeat existing observations or create synthetic ones. It avoids discarding majority data. Repeated observations can increase overfitting risk.

### Business Example
An insurance company has few confirmed claims involving fraud. It can increase the minority examples used for training. The model then sees rare cases more often. Validation and test data should stay unchanged.

### AI/ML Use
Oversampling can help a model pay attention to rare classes. Use it only on training data. Try it alongside class weights and other approaches. Evaluate whether minority recall improves without excessive false alarms.

### Python Example

In [11]:
minority = y[y == 1]
extra_ids = resample(minority.index, replace=True, n_samples=int((y == 0).sum()), random_state=42)
print(pd.concat([y[y == 0], y.loc[extra_ids]]).value_counts())

Fraud
0    216
1    216
Name: count, dtype: int64


### Explanation
The code samples minority rows with replacement. Some rows appear more than once. The new training distribution is balanced. Duplicated minority rows do not provide entirely new information.

## 6. Random Oversampling

### Definition
Random oversampling duplicates minority-class rows at random. It is simple to understand and implement. It can balance training classes without losing majority rows. It may memorize duplicated minority observations.

### Business Example
A hospital has relatively few positive screening examples. Random oversampling repeats some positive training cases. This can increase their influence during learning. Repeating cases does not guarantee better generalization.

### AI/ML Use
Start with random oversampling as a simple baseline. Perform it after the train/test split. Never duplicate rows into both train and test. Compare recall, precision, and F1 against the baseline.

### Python Example

In [13]:
minority_rows = X.loc[y == 1]
repeated = minority_rows.sample(n=5, replace=True, random_state=42)
print("Original minority rows:", len(minority_rows))
print("Extra sampled rows:", len(repeated))

Original minority rows: 24
Extra sampled rows: 5


### Explanation
The sample method draws five minority rows with replacement. Some selected rows may repeat. These can be appended to the training set if needed. The test set must never be oversampled.

## 7. Random Undersampling

### Definition
Random undersampling removes randomly selected majority rows. It is quick and reduces training size. It is suitable when majority examples are plentiful. Important majority patterns can be lost.

### Business Example
A payment processor has millions of legitimate transactions. Keeping every legitimate example may be expensive. Random undersampling can reduce training cost. The company should check whether rare legitimate patterns remain represented.

### AI/ML Use
Use it when computation or storage is limited. Compare several random seeds to check stability. Do not undersample validation or test data. Watch for reduced precision or unstable results.

### Python Example

In [15]:
keep_majority = X.loc[y == 0].sample(n=20, random_state=42)
keep_minority = X.loc[y == 1]
print("Majority kept:", len(keep_majority))
print("Minority kept:", len(keep_minority))

Majority kept: 20
Minority kept: 24


### Explanation
Only twenty majority rows are selected here. All minority rows are retained. This makes the training subset smaller. It also shows why random undersampling may lose information.

## 8. SMOTE

### Definition
SMOTE stands for Synthetic Minority Over-sampling Technique. It creates synthetic minority examples using nearby minority points. It does not merely copy rows. Synthetic observations may be unrealistic when features are unsuitable.

### Business Example
A lender has relatively few default examples. SMOTE can create new training points between similar default cases. This may help the model learn a broader decision region. Generated examples still need careful evaluation.

### AI/ML Use
SMOTE is useful for suitable numeric feature spaces. Apply it only within training folds or a training pipeline. Standardize distance-sensitive features when appropriate. For categorical features, consider specialized methods such as SMOTENC.

### Python Example

In [17]:
from imblearn.over_sampling import SMOTE
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.25, random_state=42)
X_smote, y_smote = SMOTE(random_state=42, k_neighbors=3).fit_resample(X_train, y_train)
print("Before:", y_train.value_counts().to_dict())
print("After:", y_smote.value_counts().to_dict())

Before: {0: 162, 1: 18}
After: {0: 162, 1: 162}


### Explanation
SMOTE is fitted only to the training portion. It generates new minority training samples. The class counts become balanced. The untouched test set still reflects the original distribution.

## 9. Borderline-SMOTE

### Definition
Borderline-SMOTE focuses on minority samples near the class boundary. These cases are often harder to classify. It creates synthetic examples around difficult minority regions. Noisy boundary points may still cause problems.

### Business Example
Fraudulent and genuine payments can look similar. The difficult cases lie near the boundary between classes. Borderline-SMOTE emphasizes these cases. This can help, but may also amplify noisy labels.

### AI/ML Use
Try it when boundary classification is difficult. Use training data only. Compare it against regular SMOTE and class weights. Evaluate false positives as well as minority recall.

### Python Example

In [19]:
from imblearn.over_sampling import BorderlineSMOTE
X_border, y_border = BorderlineSMOTE(random_state=42, k_neighbors=3).fit_resample(X_train, y_train)
print("Borderline-SMOTE counts:", y_border.value_counts().to_dict())

Borderline-SMOTE counts: {0: 162, 1: 162}


### Explanation
The algorithm identifies minority points near majority points. It creates additional synthetic minority examples. The printed counts show the new training balance. Always validate whether this actually improves predictions.

## 10. Class Weights

### Definition
Class weights assign different penalties to prediction errors by class. Higher weight can be given to the minority class. This avoids duplicating or removing observations. Large weights may increase false positives.

### Business Example
A bank cares strongly about missing fraudulent transactions. It can penalize missed fraud cases more heavily during training. The model may detect more fraud. It may also flag more genuine transactions.

### AI/ML Use
Use class weights when the algorithm supports them. They are a good baseline for logistic regression and many tree models. Tune weights using validation data. Do not judge success from accuracy alone.

### Python Example

In [21]:
model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
model.fit(X_train, y_train)
print("Balanced accuracy:", round(balanced_accuracy_score(y_test, model.predict(X_test)), 3))

Balanced accuracy: 0.907


### Explanation
The balanced setting increases the importance of the minority class. No training rows are added or removed. The model is evaluated on untouched test data. Balanced accuracy reflects both classes.

## 11. When to Use Each Technique

### Definition
No single balancing method works best for every dataset. The choice depends on dataset size, feature types, noise, and error costs. A simple baseline should be trained first. All resampling must happen inside training only.

### Business Example
A fraud team might start with class weights. With millions of transactions, undersampling may speed up experiments. With suitable numeric features, SMOTE may help. Results should be compared using realistic test data.

### AI/ML Use
Use class weights when supported and you want to retain all data. Try oversampling for small minority classes and undersampling for huge majority classes. Try SMOTE for suitable continuous features and Borderline-SMOTE for difficult boundaries. Use precision, recall, F1, PR-AUC, and business costs to choose.

### Python Example

In [23]:
choices = {"Class weights": "Keep all rows", "Undersampling": "Very large majority",
           "Random oversampling": "Simple minority baseline", "SMOTE": "Suitable numeric features"}
for method, use in choices.items():
    print(method, "→", use)

Class weights → Keep all rows
Undersampling → Very large majority
Random oversampling → Simple minority baseline
SMOTE → Suitable numeric features


### Explanation
This short comparison summarizes practical choices. It is guidance, not a strict rule. Test more than one method on the same training split. Never use test-set results to choose preprocessing settings.